In [10]:
from dotenv import load_dotenv
import os 

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [11]:
import requests
from langchain_core.tools import tool , InjectedToolArg
from typing import Annotated

In [14]:
# tool create
@tool 
def get_conversion_factor(base_currency: str , target_currency: str) -> float:
       """This function fetches the currency conversion factor between a given base currency and a target currency."""
       url = f"https://v6.exchangerate-api.com/v6/e3fa2998d7dc2b8305cdb3f9/pair/{base_currency}/{target_currency}"

       response = requests.get(url)

       return response.json()


@tool
def convert(base_currency_value:int , conversion_rate: Annotated[float , InjectedToolArg]) -> float:
       """Given a currency conversion rate this function calculates the target currency value from a given currency value"""
       return base_currency_value * conversion_rate


In [15]:
get_conversion_factor.invoke({'base_currency': 'USD' , 'target_currency': 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1782259202,
 'time_last_update_utc': 'Wed, 24 Jun 2026 00:00:02 +0000',
 'time_next_update_unix': 1782345602,
 'time_next_update_utc': 'Thu, 25 Jun 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.8137}

In [16]:
convert.invoke({'base_currency_value':25 , 'conversion_rate':94.4889})

2362.2225

In [17]:
# tool binding 
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id = "meta-llama/Llama-3.3-70B-Instruct" ,
    task = "text-generation"
)

model = ChatHuggingFace(llm = llm)


In [18]:
llm_with_tools = model.bind_tools([get_conversion_factor , convert])

In [19]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [20]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [21]:
ai_message = llm_with_tools.invoke(messages)

In [22]:
messages.append(ai_message)

In [23]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': '8x0zbcp2t',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'wfe8scx63',
  'type': 'tool_call'}]

In [24]:
import json 

for tool_call in ai_message.tool_calls:
   # execute the first tool and get the value of the conversion rate
   if tool_call['name'] == 'get_conversion_factor':
      tool_message1 = get_conversion_factor.invoke(tool_call)
      print(tool_message1)
      # fetch this conversion rate 
      conversion_rate = json.loads(tool_message1.content)['conversion_rate']
      # append this tool message to message list
      messages.append(tool_message1)
   # execute the second tool using the conversion from the tool1 
   if tool_call['name'] == 'convert':
      # fetch the current arg
      tool_call['args']['conversion_rate'] = conversion_rate
      tool_message2 = convert.invoke(tool_call)
      messages.append(tool_message2)




content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1782259202, "time_last_update_utc": "Wed, 24 Jun 2026 00:00:02 +0000", "time_next_update_unix": 1782345602, "time_next_update_utc": "Thu, 25 Jun 2026 00:00:02 +0000", "base_code": "INR", "target_code": "USD", "conversion_rate": 0.01055}' name='get_conversion_factor' tool_call_id='8x0zbcp2t'


In [25]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor', 'description': None}, 'id': '8x0zbcp2t', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert', 'description': None}, 'id': 'wfe8scx63', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 348, 'total_tokens': 386}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct', 'system_fingerprint': 'fp_ba38bbab80', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef876-b898-75c0-858d-c236e0631588-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': '8x0zbcp2t', 'type': 'tool_call'}, {'name': 'convert

In [26]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 0.0133. \n10 INR is equivalent to 0.1055 USD.'